# 04 Can't pay versus won't pay

Bound the number of non-paying households that could afford to pay, using three assumptions about who the non-payers are.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[2]      # repo root, two levels above this folder
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.common.config import load_config
from src.common.pipeline import build

cfg = load_config()
MODE = os.environ.get("DATA_MODE")          # None = use config; "synthetic" = fake
ghs, ies, nfcm = build(MODE, save=False)

In [ ]:
from src.a_who_pays import split

nonpayers = split.nonpayers_by_province(ghs)
nonpayers

In [ ]:
rows = []
for t in cfg["thresholds"]["bill_to_income"]:
    bounds = split.unable_bounds(ies, nonpayers, t)
    rows.append(split.split_summary(bounds, t))
summary = pd.DataFrame(rows)
summary

Report the low-to-high range. The mid case assumes non-payers look like all households with a bill.

In [ ]:
labels = [f"{t * 100:.0f}% threshold" for t in summary["threshold"]]
mid = summary["able_share_mid"] * 100
low = mid - summary["able_share_low"] * 100
high = summary["able_share_high"] * 100 - mid
plt.errorbar(mid, labels, xerr=[low, high], fmt="o", capsize=5)
plt.xlabel("Non-paying households able to pay (%)")
plt.show()